# 🚀 Engenharia de Recursos, Modelagem e MLflow Registry

Este notebook implementa a vetorização TF-IDF dos resumos médicos limpos, o treinamento do modelo de classificação baseline (Regressão Logística) e o registro formal do artefato utilizando o MLflow Model Registry, garantindo o versionamento e a promoção solicitados para o Tech Challenge.

### ⚙️ 1. Vetorização TF-IDF e Treinamento do Modelo
Este bloco carrega os dados limpos gerados na EDA, converte os textos em uma matriz numérica com o TfidfVectorizer e treina o modelo de Regressão Logística.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

# Carregar os dados limpos da etapa anterior
train_df = pd.read_csv("../data/processed/medical_tc_train_clean.csv")

# 1. Vetorização TF-IDF
print("A aplicar vetorização TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["clean_abstract"].fillna(""))
y_train = train_df["condition_label"]

# 2. Treinamento do Modelo Baseline
print("A treinar Regressão Logística...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred_train = model.predict(X_train_tfidf)
train_acc = accuracy_score(y_train, y_pred_train)
print(f"Acurácia de Treino: {train_acc:.4f}")

A aplicar vetorização TF-IDF...
A treinar Regressão Logística...
Acurácia de Treino: 0.7165


### 📊 2. Rastreamento e Registro no MLflow Model Registry
Este bloco configura o experimento no MLflow, registra os parâmetros e métricas, e envia o modelo diretamente para o Model Registry usando o parâmetro registered_model_name.

In [3]:
import mlflow
import mlflow.sklearn

# Configurar o experimento e registrar o modelo formalmente
mlflow.set_experiment("medical-triage-classification")

with mlflow.start_run(run_name="logistic-regression-registry-baseline") as run:
  # Parâmetros e Métricas
  mlflow.log_param("vectorizer", "TfidfVectorizer")
  mlflow.log_param("max_features", 5000)
  mlflow.log_param("model_type", "LogisticRegression")
  mlflow.log_metric("train_accuracy", train_acc)

  # Log e Registro formal do Modelo no Registry
  model_name = "MedicalTriageClassifier"
  
  mlflow.sklearn.log_model(
      sk_model=model,
      artifact_path="model",
      registered_model_name=model_name
  )
  
  print(f"\nModelo '{model_name}' registrado com sucesso no MLflow Model Registry!")
  print(f"Run ID: {run.info.run_id}")

2026/09/09 21:08:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Modelo 'MedicalTriageClassifier' registrado com sucesso no MLflow Model Registry!
Run ID: 4ce3f0e70c8d4de8937d623df23008f6


Registered model 'MedicalTriageClassifier' already exists. Creating a new version of this model...
Created version '2' of model 'MedicalTriageClassifier'.


### 📈 3. Avaliação de Desempenho e Validação com Dados de Teste
Este bloco carrega o conjunto de teste, aplica o mesmo vetorizador treinado e avalia a capacidade de generalização do modelo utilizando um relatório de classificação completo.

In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Carregar os dados de teste limpos
test_df = pd.read_csv("../data/processed/medical_tc_test_clean.csv")

# Aplicar a transformação TF-IDF nos dados de teste (apenas transform, sem fit!)
X_test_tfidf = tfidf.transform(test_df["clean_abstract"].fillna(""))
y_test = test_df["condition_label"]

# Realizar previsões
y_pred_test = model.predict(X_test_tfidf)

# Avaliar métricas
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Acurácia de Teste: {test_accuracy:.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred_test))

Acurácia de Teste: 0.5699

Relatório de Classificação:
              precision    recall  f1-score   support

           1       0.69      0.70      0.69       633
           2       0.54      0.39      0.46       299
           3       0.58      0.44      0.50       385
           4       0.65      0.65      0.65       610
           5       0.46      0.54      0.50       961

    accuracy                           0.57      2888
   macro avg       0.58      0.54      0.56      2888
weighted avg       0.58      0.57      0.57      2888

